In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
# Nếu cần, set lại catalog/schema giống lúc tạo bảng
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA health")

# Đọc bảng Gold chính làm dữ liệu ML
df = spark.table("gold_mart_patient_day")

df.printSchema()
df.show(5)
print("Total rows:", df.count())

In [0]:
# Load table gold_mart_patient_day vào Spark DF
spark_df = spark.table("gold_mart_patient_day")

# Convert sang pandas (chỉ nên làm nếu dữ liệu ~ vài chục nghìn dòng)
pdf = spark_df.toPandas()

pdf.head()


In [0]:
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [0]:
print(pdf.shape)
print(pdf.dtypes)
pdf.describe().T


Kiểm tra Missing Values

In [0]:
pdf.isna().sum()

sns.heatmap(pdf.isna(), cbar=False)
plt.title("Missing Data Heatmap")
plt.show()


EDA UNIVARIATE

In [0]:
sns.histplot(pdf["avg_hr"], kde=True, bins=40)
plt.title("Distribution of Heart Rate (avg_hr)")
plt.show()


In [0]:
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

sns.histplot(pdf["avg_sys"], kde=True, ax=ax[0], color='blue')
ax[0].set_title("Systolic BP Distribution")

sns.histplot(pdf["avg_dia"], kde=True, ax=ax[1], color='red')
ax[1].set_title("Diastolic BP Distribution")

plt.show()


In [0]:
sns.histplot(pdf["avg_spo2"], kde=True)
plt.title("Oxygen Saturation Distribution")
plt.show()


In [0]:
cols = ["cholesterol", "glucose", "hemoglobin"]

for c in cols:
    sns.histplot(pdf[c], kde=True, bins=40)
    plt.title(f"Distribution of {c}")
    plt.show()


EDA CATEGORY (risk_label, gender, region)

In [0]:
sns.countplot(data=pdf, x="risk_label", order=["Normal", "Elevated", "High"])
plt.title("Risk Label Distribution")
plt.show()


In [0]:
sns.countplot(data=pdf, x="gender", hue="risk_label")
plt.title("Risk Label by Gender")
plt.show()


In [0]:
sns.countplot(data=pdf, x="region", hue="risk_label")
plt.title("Risk Label by Region")
plt.show()


MULTIVARIATE (Quan hệ giữa các biến)

In [0]:
sns.jointplot(data=pdf, x="avg_sys", y="avg_dia", kind="hex")
plt.suptitle("BP Relationship (SYS vs DIA)")
plt.show()


In [0]:
sns.scatterplot(data=pdf, x="avg_hr", y="avg_spo2", hue="risk_label")
plt.title("HR vs SPO2 by Risk Label")
plt.show()


In [0]:
numeric_cols = [
    "avg_hr", "avg_sys", "avg_dia", "avg_spo2",
    "cholesterol", "glucose", "hemoglobin",
    "age_est", "total_flags", "risk_score"
]

corr = pdf[numeric_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap (Numeric Features Only)")
plt.show()
